# Gravity Field Computation — Top-Shaped Asteroid

Computes the gravitational acceleration field on the surface of a
double-cone body representing Bennu by integrating over stacked discs.
Writes `grav45.txt` and `eta45.txt` for use by `bennu_solver.cpp`.

**Run this notebook before the C++ solver.**

In [1]:
import numpy as np
import scipy.special
import mpmath as mp
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

ModuleNotFoundError: No module named 'numpy'

## Parameters

`n_surface` must match the compile-time `RES` in `config.h` so the
output files have the correct number of rows.
`n_integ` controls integration accuracy (more steps = slower but better).

In [ ]:
n_surface = 200       # surface grid points — must match config.h RES
n_integ   = 100       # integration steps per half-height (accuracy control)
apex_deg  = 45.0      # cone half-apex angle (degrees)
omega_plot = 1.5      # spin rate used for effective gravity plots
mu        = np.tan(20.0 * np.pi / 180.0)   # Coulomb friction coefficient

## Geometry

The double-cone has a normalised slant height of 1.  
`Tau` is the radial coordinate from pole to equator.  
`coordZ` is the corresponding axial coordinate.

In [ ]:
zeta   = apex_deg * np.pi / 180.0
HE     = np.cos(zeta)     # axial half-height of one cone
RAD    = np.sin(zeta)     # equatorial radius

# Cell-centred surface coordinates (pole -> equator)
coordZ = np.linspace(HE - HE / n_surface / 2, HE / n_surface / 2, n_surface)
Tau    = np.linspace(RAD / n_surface / 2, RAD - RAD / n_surface / 2, n_surface)

print(f'Apex half-angle : {apex_deg:.1f} deg')
print(f'Axial half-height HE  = {HE:.4f}')
print(f'Equatorial radius RAD = {RAD:.4f}')
print(f'Surface points: {n_surface},  integration steps per point: {2*n_integ+1}')

## Gravity Integrand Functions

Each thin disc at axial position `z` contributes to the radial and axial
gravity at surface point `(R, Z)`, expressed via complete elliptic integrals.

In [ ]:
def disc_gravity_radial(z, R, Z):
    """Radial gravity contribution from a thin disc at axial position z."""
    a     = (HE - abs(z)) * np.tan(zeta)
    dz    = Z - z
    delta = np.sqrt((a + R)**2 + dz**2)
    k     = 2.0 * np.sqrt(a * R) / delta
    K     = scipy.special.ellipk(k**2)
    E     = scipy.special.ellipe(k**2)
    return 2.0 * delta * ((1.0 - k**2 / 2.0) * K - E) / R


def disc_gravity_axial(z, R, Z):
    """Axial gravity contribution from a thin disc at axial position z."""
    a = (HE - abs(z)) * np.tan(zeta)
    if   R < a: eps = 1.0
    elif R > a: eps = 0.0
    else:       eps = 0.5
    dz    = Z - z
    delta = np.sqrt((a + R)**2 + dz**2)
    k     = 2.0 * np.sqrt(a * R) / delta
    m     = 2.0 * np.sqrt(a * R) / (a + R)
    K     = scipy.special.ellipk(k**2)
    Pi    = mp.ellippi(m**2, k**2)
    return (2.0 * np.pi * np.sign(dz) * eps
            + 2.0 * dz * ((R - a) / (R + a) * Pi - K) / delta)


def disc_potential(z, R, Z):
    """Gravitational potential contribution from a thin disc at axial position z."""
    a = (HE - abs(z)) * np.tan(zeta)
    if   R < a: eps = 1.0
    elif R > a: eps = 0.0
    else:       eps = 0.5
    dz    = Z - z
    delta = np.sqrt((a + R)**2 + dz**2)
    k     = 2.0 * np.sqrt(a * R) / delta
    m     = 2.0 * np.sqrt(a * R) / (a + R)
    K     = scipy.special.ellipk(k**2)
    E     = scipy.special.ellipe(k**2)
    Pi    = mp.ellippi(m**2, k**2)
    return 2.0 * (-np.pi * abs(dz) * eps + delta * E
                  + (a**2 - R**2) * K / delta
                  + dz**2 / delta * (-R + a) / (R + a) * Pi)

## Numerical Integration Over the Full Double-Cone

Sums disc contributions from `-HE` to `+HE` (both cone halves).
With `n_surface=200` and `n_integ=100` this may take a few minutes.

In [ ]:
def integrate_gravity(n, R, Z, flag):
    """
    Riemann sum over n_integ discs per half-cone.
    flag: 1=radial gravity, 2=axial gravity, 3=potential.
    """
    dz    = HE / n
    total = 0.0
    for i in range(-n, n + 1):
        z = i * dz
        if   flag == 1: total += dz * disc_gravity_radial(z, R, Z)
        elif flag == 2: total += dz * disc_gravity_axial(z, R, Z)
        else:           total += dz * disc_potential(z, R, Z)
    return total

## Compute Gravity Field (main loop)

In [ ]:
MR  = np.zeros(n_surface)
MZ  = np.zeros(n_surface)
MAG = np.zeros(n_surface)
ETA = np.zeros(n_surface)
POT = np.zeros(n_surface)

for l in range(n_surface):
    POT[l] = integrate_gravity(n_integ, Tau[l], coordZ[l], 3)
    MR[l]  = abs(integrate_gravity(n_integ, Tau[l], coordZ[l], 1))
    MZ[l]  = abs(integrate_gravity(n_integ, Tau[l], coordZ[l], 2))
    MAG[l] = np.sqrt(MR[l]**2 + MZ[l]**2)
    ETA[l] = np.arctan2(MR[l], MZ[l])   # angle from axial direction (rad)
    if l % 20 == 0:
        print(f'  {l}/{n_surface} surface points computed...')

MAG = MAG / MAG[-1]   # normalise to equatorial value
print('Done.')

## Plot: Gravity Magnitude and Direction

In [ ]:
s = Tau / Tau[-1]   # normalised downslope coordinate (0=pole, 1=equator)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(s, MAG, linewidth=2)
axes[0].set_xlabel('Normalised downslope coordinate s')
axes[0].set_ylabel('Normalised gravity magnitude')
axes[0].set_title('Gravity magnitude along slope')
axes[0].grid(alpha=0.3)

axes[1].plot(s, np.degrees(ETA), linewidth=2, color='C1')
axes[1].set_xlabel('Normalised downslope coordinate s')
axes[1].set_ylabel('Gravity angle eta (degrees from axial)')
axes[1].set_title('Gravity vector direction')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Effective Gravity with Rotation

Gravitational acceleration projected onto slope-tangential (downslope)
and slope-normal directions, plus centrifugal contribution.

In [ ]:
MTAN = np.zeros(n_surface)   # gravity: tangential (downslope) component
MNOR = np.zeros(n_surface)   # gravity: normal (into slope) component
ETAN = np.zeros(n_surface)   # effective tangential (gravity + centrifugal)
ENOR = np.zeros(n_surface)   # effective normal
VESC = np.zeros(n_surface)   # escape velocity

for l in range(n_surface):
    MTAN[l] = MAG[l] * np.cos(zeta + ETA[l])
    MNOR[l] = MAG[l] * np.sin(zeta + ETA[l])
    VESC[l] = np.sqrt(2.0 * POT[l])
    ETAN[l] = MTAN[l] + omega_plot**2 * Tau[l] * np.sin(zeta)
    ENOR[l] = -MNOR[l] + omega_plot**2 * Tau[l] * np.cos(zeta)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(s, MTAN, label='Tangential (downslope)', linewidth=2)
axes[0].plot(s, MNOR, label='Normal (into slope)',    linewidth=2)
axes[0].axhline(0, color='k', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Normalised downslope coordinate s')
axes[0].set_ylabel('Acceleration (normalised)')
axes[0].set_title('Gravitational components on slope')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(s, ETAN, label=f'Effective tangential (omega={omega_plot})', linewidth=2)
axes[1].plot(s, ENOR, label='Effective normal', linewidth=2)
axes[1].axhline(0, color='k', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Normalised downslope coordinate s')
axes[1].set_ylabel('Acceleration (normalised)')
axes[1].set_title('Effective gravity (gravity + centrifugal)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Stability Analysis: Critical Angular Velocities

- **ONOR**: spin rate for zero effective normal acceleration (takeoff threshold)
- **OTAN**: spin rate for zero effective tangential acceleration (slope equilibration)
- **OMP**: spin bounds for static equilibrium including Coulomb friction

In [ ]:
import math

ONOR = np.zeros(n_surface)
OTAN = np.full(n_surface, np.nan)
OMP  = np.full((2, n_surface), np.nan)
VLO  = np.zeros((2, n_surface))

for l in range(n_surface - 1):
    # Takeoff: zero normal effective gravity
    arg = MNOR[l] / np.cos(zeta) / Tau[l]
    if arg >= 0:
        ONOR[l] = np.sqrt(arg)

    # Slope equilibration: zero tangential effective gravity
    arg = -MTAN[l] / np.sin(zeta) / Tau[l]
    if arg >= 0:
        OTAN[l] = np.sqrt(arg)
    elif (-MTAN[l + 1] / np.sin(zeta) / Tau[l + 1]) >= 0:
        OTAN[l] = 0.0

    # Friction equilibrium bounds (upper and lower omega)
    for k, sign in enumerate([1, -1]):
        num = MTAN[l] + sign * mu * MNOR[l]
        den = Tau[l] * (sign * mu * np.cos(zeta) - np.sin(zeta))
        if abs(den) > 1e-12:
            arg = num / den
            if arg >= 0:
                OMP[k, l] = np.sqrt(arg)
            elif l + 1 < n_surface:
                n1 = MTAN[l+1] + sign * mu * MNOR[l+1]
                d1 = Tau[l+1] * (sign * mu * np.cos(zeta) - np.sin(zeta))
                if abs(d1) > 1e-12 and n1 / d1 >= 0:
                    OMP[k, l] = 0.0

    # Lift-off azimuthal velocity
    disc = (omega_plot * Tau[l])**2 - Tau[l] * ENOR[l] / np.cos(zeta)
    if disc >= 0:
        VLO[0, l] = -omega_plot * Tau[l] - np.sqrt(disc)
        VLO[1, l] = -omega_plot * Tau[l] + np.sqrt(disc)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(s, ONOR,      label='Takeoff omega',        linewidth=2)
axes[0].plot(s, OTAN,      label='Slope equil. omega',   linewidth=2)
axes[0].plot(s, OMP[0, :], label='Friction upper omega', linewidth=2, linestyle='--')
axes[0].plot(s, OMP[1, :], label='Friction lower omega', linewidth=2, linestyle='--')
axes[0].set_xlabel('Normalised downslope coordinate s')
axes[0].set_ylabel('Critical spin rate (non-dimensional)')
axes[0].set_title('Critical angular velocities')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 4)
axes[0].grid(alpha=0.3)

axes[1].plot(s, VLO[0, :], label='Lift-off v (lower)',     linewidth=2)
axes[1].plot(s, VLO[1, :], label='Lift-off v (upper)',     linewidth=2)
axes[1].plot(s, ENOR,      label='Effective normal accel', linewidth=2, linestyle=':')
axes[1].axhline(0, color='k', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Normalised downslope coordinate s')
axes[1].set_ylabel('Velocity / Acceleration (normalised)')
axes[1].set_title('Lift-off velocities')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Write Gravity Field Files

Produces `grav45.txt` (normalised gravity magnitude) and `eta45.txt`
(gravity angle in radians). Each has `n_surface` rows — one value per line —
matching the C++ solver grid (RES in `config.h`).

In [ ]:
with open('grav45.txt', 'w') as f:
    for val in MAG:
        print(val, file=f)

with open('eta45.txt', 'w') as g:
    for val in ETA:
        print(val, file=g)

print(f'Written grav45.txt and eta45.txt  ({n_surface} values each).')
print(f'Gravity range : {MAG.min():.4f} to {MAG.max():.4f}')
print(f'ETA range     : {np.degrees(ETA.min()):.2f} to {np.degrees(ETA.max()):.2f} deg')